# 13 — Supervised Fraud Detection: End-to-End Classification Pipeline

**Objective:** build a production-aligned supervised fraud classifier from the raw e-commerce fraud dataset in one notebook: chronological train/validation/test split, EDA, leakage-safe feature engineering, preprocessing, Logistic Regression + XGBoost + LightGBM, GridSearchCV, imbalance-strategy comparison, validation threshold tuning, locked test evaluation, error analysis, explainability, and serialization.

**Model decision:** Random Forest and Extra Trees are removed. The candidate families are **Logistic Regression, XGBoost, and LightGBM**.

**Deployment rule:** the saved fitted pipeline receives the same raw transaction columns used during training. Streamlit must not manually recreate preprocessing or feature engineering.

## 1. Imports and reproducibility
We use a fixed seed. The supervised module contains the complete feature-engineering and inference pipeline.

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

ROOT = Path.cwd().resolve().parents[0]
if not (ROOT / "data").exists():
    ROOT = Path.cwd().resolve()

SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

from supervised_fraud import (
    BASE_REQUIRED,
    TARGET,
    FraudFeatureTransformer,
    make_preprocessor,
    build_searches,
    evaluate_model,
    select_threshold,
    save_artifact,
)

SEED = 42
DATA_PATH = ROOT / "data/raw/Fraud_Data.csv"
IP_PATH = ROOT / "data/raw/IpAddress_to_Country.csv"

warnings.filterwarnings("ignore")
np.random.seed(SEED)

print("ROOT:", ROOT)
print("Data:", DATA_PATH)
print("IP map:", IP_PATH)

## 2. Load raw data and audit the target

In [ ]:
df = pd.read_csv(DATA_PATH)
ip_map = pd.read_csv(IP_PATH)

print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))

print("\nFraud distribution:")
display(
    df[TARGET]
    .value_counts()
    .rename_axis(TARGET)
    .to_frame("count")
    .assign(rate=lambda x: x["count"] / len(df))
)

missing = df.isna().sum().sort_values(ascending=False)
display(missing.to_frame("missing_count").query("missing_count > 0"))

print("Duplicate rows:", df.duplicated().sum())
print("Fraud rate: {:.3%}".format(df[TARGET].mean()))

## 3. Chronological train / validation / locked test split

Fraud behavior can drift over time, so the split is chronological using `purchase_time`.

- **70% earliest:** training
- **15% next:** validation
- **15% latest:** locked test

The test set is not used for model selection or threshold selection.

In [ ]:
df["signup_time"] = pd.to_datetime(df["signup_time"], errors="coerce", utc=True)
df["purchase_time"] = pd.to_datetime(df["purchase_time"], errors="coerce", utc=True)

df = df.sort_values("purchase_time").reset_index(drop=True)

train_end = df["purchase_time"].quantile(0.70)
val_end = df["purchase_time"].quantile(0.85)

train = df[df.purchase_time <= train_end].copy()
val = df[(df.purchase_time > train_end) & (df.purchase_time <= val_end)].copy()
test = df[df.purchase_time > val_end].copy()

print("Train:", train.shape, "fraud rate:", train[TARGET].mean())
print("Validation:", val.shape, "fraud rate:", val[TARGET].mean())
print("Test:", test.shape, "fraud rate:", test[TARGET].mean())
print("Train end:", train_end)
print("Validation end:", val_end)

## 4. EDA for model-design decisions

In [ ]:
eda = train.copy()
eda["account_age_hours"] = (
    eda["purchase_time"] - eda["signup_time"]
).dt.total_seconds() / 3600

fig, ax = plt.subplots(figsize=(8, 4))
for label in sorted(eda[TARGET].dropna().unique()):
    values = eda.loc[eda[TARGET] == label, "purchase_value"].dropna()
    ax.hist(
        values,
        bins=60,
        density=True,
        histtype="step",
        linewidth=1.5,
        label=f"{TARGET}={int(label)}",
    )
ax.set_title("Purchase value distribution by target")
ax.set_xlabel("Purchase value")
ax.set_ylabel("Density")
ax.legend()
plt.show()

hour = (
    train.assign(hour=train.purchase_time.dt.hour)
    .groupby("hour")[TARGET]
    .agg(["mean", "count"])
    .reset_index()
)
display(hour)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hour["hour"], hour["mean"], marker="o")
ax.set_title("Fraud rate by purchase hour — training data")
ax.set_xlabel("Purchase hour")
ax.set_ylabel("Fraud rate")
plt.show()

## 5. Leakage-safe feature engineering

Raw high-cardinality identifiers are not one-hot encoded. Instead, the transformer derives frequency and relationship features for user/device/IP.

All frequency statistics are learned from the data passed to `fit()`. Because the transformer is inside the GridSearchCV pipeline, each cross-validation fold learns these statistics from its own training fold only.

In [ ]:
feature_demo = FraudFeatureTransformer(ip_map=ip_map).fit(
    train.drop(columns=[TARGET])
)
train_features = feature_demo.transform(train.drop(columns=[TARGET]))

print("Engineered columns:")
print(train_features.columns.tolist())
display(train_features.head())

## 6. Preprocessing

In [ ]:
preprocessor = make_preprocessor()
print(preprocessor)

## 7. Candidate models + imbalance strategies + GridSearchCV

We compare **three model families**:

1. Logistic Regression — fast linear baseline
2. XGBoost — gradient-boosted trees with histogram training
3. LightGBM — gradient-boosted trees optimized for tabular data

For each model we compare:

1. **none** — natural class distribution
2. **class_weight** — native positive-class weighting (`scale_pos_weight` for XGBoost)
3. **SMOTE** — synthetic minority oversampling inside each CV training fold

**Primary metric:** average precision (PR-AUC).

The grid is intentionally focused so that every model and every imbalance strategy gets a real GridSearchCV experiment without repeating the very large Random Forest / Extra Trees search that previously took hours.

In [ ]:
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]

print("Training rows:", len(X_train))
print("Training fraud rate:", y_train.mean())

searches = build_searches(
    ip_map=ip_map,
    seed=SEED,
    cv_splits=3,
    n_jobs=-1,
    y_train=y_train,
)

print("GridSearch experiments:", len(searches))
print("\nExperiments:")
for name in searches:
    print(" -", name)

In [ ]:
# Run every model × imbalance-strategy GridSearch.
# Each search uses 3-fold CV and PR-AUC as the scoring objective.
search_results = {}

for i, (name, search) in enumerate(searches.items(), start=1):
    print(f"[{i}/{len(searches)}] Running: {name}")
    search.fit(X_train, y_train)
    search_results[name] = search
    print(f"    Best CV PR-AUC: {search.best_score_:.6f}")
    print(f"    Best params: {search.best_params_}")

print(f"\nCompleted {len(search_results)} GridSearchCV experiments.")

## 8. Compare all experiments on the validation set

The CV score tells us how the candidate performed inside the training folds. Final model/strategy selection is based on **validation PR-AUC**, with CV PR-AUC retained as supporting evidence.

We do not use the locked test set for this decision.

In [ ]:
rows = []

X_val = val.drop(columns=[TARGET])
y_val = val[TARGET]

for name, search in search_results.items():
    proba = search.predict_proba(X_val)[:, 1]
    model_name, strategy = name.split(" | ")

    rows.append({
        "model": model_name,
        "strategy": strategy,
        "cv_pr_auc": search.best_score_,
        "validation_pr_auc": average_precision_score(y_val, proba),
        "validation_roc_auc": roc_auc_score(y_val, proba),
        "best_params": search.best_params_,
    })

comparison = (
    pd.DataFrame(rows)
    .sort_values(
        ["validation_pr_auc", "cv_pr_auc"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    comparison[
        [
            "model",
            "strategy",
            "cv_pr_auc",
            "validation_pr_auc",
            "validation_roc_auc",
            "best_params",
        ]
    ]
)

print("\nBest experiment by validation PR-AUC:")
display(comparison.head(1))

## 9. Select the best model + balancing strategy and tune the operating threshold

The winning pipeline is selected using validation PR-AUC. The classification threshold is then selected **only on validation data** using maximum F1 by default.

The probability ranking metric (PR-AUC) is independent of the final 0.5/default classification threshold.

In [ ]:
best_row = comparison.iloc[0]
best_name = f"{best_row['model']} | {best_row['strategy']}"
best_search = search_results[best_name]
best_pipeline = best_search.best_estimator_

val_proba = best_pipeline.predict_proba(X_val)[:, 1]

validation_metrics_at_05 = evaluate_model(
    best_pipeline,
    X_val,
    y_val,
    threshold=0.5,
)

threshold = select_threshold(
    y_val.to_numpy(),
    val_proba,
)

validation_metrics_tuned = evaluate_model(
    best_pipeline,
    X_val,
    y_val,
    threshold=threshold,
)

print("Selected model:", best_row["model"])
print("Selected balancing strategy:", best_row["strategy"])
print("Best parameters:", best_search.best_params_)
print(f"Validation PR-AUC: {average_precision_score(y_val, val_proba):.4f}")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, val_proba):.4f}")
print(f"Selected validation threshold: {threshold:.4f}")

display(pd.DataFrame([validation_metrics_tuned]))

## 10. Precision-recall threshold analysis

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_val, val_proba)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall, precision)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Validation precision-recall curve")
plt.show()

threshold_table = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1],
})
threshold_table["f1"] = (
    2 * threshold_table.precision * threshold_table.recall
    / (
        threshold_table.precision
        + threshold_table.recall
        + 1e-12
    )
)
display(
    threshold_table
    .sort_values("f1", ascending=False)
    .head(15)
)

## 11. Refit the selected pipeline on train + validation, then evaluate once on the locked test

After all model, hyperparameter, imbalance-strategy, and threshold decisions are complete, we refit the selected pipeline on **train + validation**.

The threshold remains the value selected on validation. The test set is used only now for final generalization reporting.

In [ ]:
train_val = pd.concat([train, val], ignore_index=True)

final_pipeline = best_pipeline.set_params(**best_search.best_params_)
final_pipeline.fit(
    train_val.drop(columns=[TARGET]),
    train_val[TARGET],
)

test_proba = final_pipeline.predict_proba(
    test.drop(columns=[TARGET])
)[:, 1]

test_metrics = evaluate_model(
    final_pipeline,
    test.drop(columns=[TARGET]),
    test[TARGET],
    threshold=threshold,
)

print(json.dumps(test_metrics, indent=2))

In [ ]:
pred = (test_proba >= threshold).astype(int)

print(classification_report(test[TARGET], pred, digits=4))

cm = confusion_matrix(test[TARGET], pred)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, interpolation="nearest")
ax.set_title("Locked test confusion matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.show()

RocCurveDisplay.from_predictions(test[TARGET], test_proba)
plt.title("Locked test ROC curve")
plt.show()

PrecisionRecallDisplay.from_predictions(test[TARGET], test_proba)
plt.title("Locked test PR curve")
plt.show()

## 12. Error analysis

In [ ]:
errors = test.copy()
errors["fraud_probability"] = test_proba
errors["prediction"] = pred
errors["error_type"] = np.select(
    [
        (errors[TARGET] == 1) & (pred == 0),
        (errors[TARGET] == 0) & (pred == 1),
    ],
    ["False Negative", "False Positive"],
    default="Correct",
)

display(
    errors[errors.error_type != "Correct"][
        [
            "user_id",
            "purchase_value",
            "source",
            "browser",
            "age",
            "fraud_probability",
            TARGET,
            "prediction",
            "error_type",
        ]
    ]
    .sort_values("fraud_probability", ascending=False)
    .head(50)
)

## 13. Explainability / feature importance

Permutation importance is calculated on a validation sample using PR-AUC. This answers which **raw input columns** most affect the model's fraud-ranking performance.

It is an explanatory diagnostic, not a causal claim.

In [ ]:
from sklearn.inspection import permutation_importance

X_exp = val.drop(columns=[TARGET]).sample(
    min(3000, len(val)),
    random_state=SEED,
)
y_exp = val.loc[X_exp.index, TARGET]

pi = permutation_importance(
    best_pipeline,
    X_exp,
    y_exp,
    scoring="average_precision",
    n_repeats=3,
    random_state=SEED,
    n_jobs=-1,
)

imp = (
    pd.DataFrame({
        "feature": X_exp.columns,
        "importance": pi.importances_mean,
    })
    .sort_values("importance", ascending=False)
)

display(imp.head(20))

## 14. Save the complete inference artifact

The artifact contains the fitted feature engineering + preprocessing + final classifier.

For the winning SMOTE experiment, SMOTE remains a training-time pipeline step and is not applied to inference rows.

**Streamlit contract:** pass raw transaction columns into this saved artifact. Do not duplicate the transformation logic in the UI.

In [ ]:
metadata = {
    "model_name": best_name,
    "best_params": best_search.best_params_,
    "validation_threshold": float(threshold),
    "test_metrics": test_metrics,
    "train_end": str(train_end),
    "validation_end": str(val_end),
    "raw_required_columns": BASE_REQUIRED,
    "target": TARGET,
    "candidate_models": [
        "LogisticRegression",
        "XGBoost",
        "LightGBM",
    ],
    "imbalance_strategies": [
        "none",
        "class_weight",
        "smote",
    ],
    "primary_selection_metric": "validation PR-AUC",
    "cv_splits": 3,
}

artifact_path = ROOT / "artifacts/fraud_classifier.joblib"
save_artifact(final_pipeline, artifact_path, metadata)

metadata_path = ROOT / "artifacts/fraud_classifier_metadata.json"
metadata_path.write_text(
    json.dumps(metadata, indent=2, default=str),
    encoding="utf-8",
)

print("Saved:", artifact_path)
print("Saved:", metadata_path)

## 15. Final experiment conclusion

This notebook compares **Logistic Regression, XGBoost, and LightGBM** under three imbalance strategies: no balancing, class weighting/native positive-class weighting, and SMOTE.

The model, hyperparameters, and imbalance strategy are selected from observed validation performance rather than assumed in advance. **Validation PR-AUC is the primary selection metric**, while recall, precision, F1, ROC-AUC, balanced accuracy, and the confusion matrix provide operating context.

The locked test set is evaluated only after all decisions are frozen.

### Portfolio takeaway

The project demonstrates:
- chronological evaluation for temporal fraud data,
- leakage-safe feature engineering,
- systematic hyperparameter search,
- explicit imbalance-strategy comparison,
- validation-only threshold tuning,
- locked-test evaluation,
- and a single serialized train-serving pipeline shared by notebook and Streamlit.